In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

### Notebook for training a decision model (logistic regression)

Bets Practice: 
Running pinarea_decision.py with an existing .joblib file produces a CSV file.
The labels can then be set manually here to obtain ground truth for training

In [ ]:
# --- Load data from CSV ---
import os

# Path to the CSV file (update if necessary)
csv_path = "decision_results.csv"

# Load the data
board_stats = pd.read_csv(csv_path)

print(f"Data loaded: {board_stats.shape[0]} rows, {board_stats.shape[1]} columns")
print(f"Columns: {list(board_stats.columns)}")
print(f"\nFirst few rows:")
print(board_stats.head())
print(f"\nLabel distribution:")
print(board_stats['label'].value_counts())
print(f"\nUnique labels: {board_stats['label'].unique()}")
print(f"\nData types:")
print(board_stats.dtypes)

In [ ]:
# --- Set Features ---
feature_cols = ['avg_flaeche', 'std_flaeche', 'max_flaeche', 'anzahl_pins', 'mean_aspect_ratio']

# Check if all feature columns exist in the DataFrame
X = board_stats[feature_cols].copy()
X = X.fillna(0.0)

# Labels: Convert 'pass' to 1 and 'fail' to 0
y = (board_stats['label'].astype(str).str.lower() == 'pass').astype(int)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Class distribution: {y.value_counts().to_dict()}")

# DATA LIMITATION NOTICE
print("\n" + "="*70)
if y.nunique() < 2:
    print("IMPORTANT: Only one class found in labels!")
    print("="*70)
    print("\nThis dataset contains only 'pass' labels and cannot be used to train")
    print("a binary classification model. A logistic regression classifier needs")
    print("examples of BOTH classes ('pass' and 'fail').")
    print("\n✓ SUGGESTIONS:")
    print("  1. Combine this dataset with one containing 'fail' samples")
    print("  2. Look for other CSV files in roboflow/inferenz/ with 'fail' labels")
    print("  3. For now, we can analyze the feature statistics of 'pass' boards")
    print("="*70)
    
    # Show descriptive statistics instead
    print("\nDescriptive Statistics for Features (Pass boards only):")
    print(board_stats[feature_cols].describe())
    
    print("\nSkipping classifier training - need both classes in training data.")
else:
    stratify_param = y
    
    # --- Train/Test Split ---
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=stratify_param
    )
    
    # --- Feature Scaling ---
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)
    
    # --- Logistic Regression ---
    logreg = LogisticRegression(
        C=1.0,
        solver='lbfgs',
        max_iter=1000,
        class_weight='balanced'
    )
    
    logreg.fit(X_train_sc, y_train)
    print("\n✓ Model trained successfully")
    
    # --- Evaluation ---
    y_pred = logreg.predict(X_test_sc)
    y_proba = logreg.predict_proba(X_test_sc)[:, 1]
    
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, digits=3))
    
    try:
        auc = roc_auc_score(y_test, y_proba)
        print(f"ROC-AUC: {auc:.3f}")
    except ValueError as e:
        print(f"ROC-AUC could not be calculated: {e}")
    
    # --- Coefficients ---
    coef_df = pd.DataFrame({
        'feature': feature_cols,
        'coef_standardized': logreg.coef_[0]
    }).sort_values('coef_standardized', ascending=False)
    print("\nStandardised Coefficients:")
    print(coef_df)
    
    # ROC-Kurve
    try:
        fig, ax = plt.subplots()
        RocCurveDisplay.from_estimator(logreg, X_test_sc, y_test, ax=ax)
        ax.set_title("ROC-Curve")
        plt.show()
    except Exception as e:
        print(f"Could not plot ROC curve: {e}")

In [ ]:
y_scores = logreg.predict_proba(X_test_sc)[:, 1]   # P(pass)

thresholds = np.linspace(0.05, 0.95, 50)
#thresholds = np.linspace(0.42, 0.44, 3) # fine tuning

results = []

for t in thresholds:
    y_pred_t = (y_scores < t).astype(int) * 0 + (y_scores >= t).astype(int)

    # 0 = fail, 1 = pass
    precision_fail = precision_score(y_test, y_pred_t, pos_label=0)
    recall_fail    = recall_score(y_test, y_pred_t,    pos_label=0)
    f1_fail        = f1_score(y_test, y_pred_t,        pos_label=0)

    # pass metrics (pos_label=1)
    precision_pass = precision_score(y_test, y_pred_t, pos_label=1, zero_division=0)
    recall_pass = recall_score(y_test, y_pred_t,    pos_label=1, zero_division=0)
    f1_pass        = f1_score(y_test, y_pred_t,        pos_label=1)

    results.append((t, precision_fail, recall_fail, f1_fail, precision_pass, recall_pass, f1_pass))

df = pd.DataFrame(results, columns=['threshold','precision_fail','recall_fail','f1_fail','precision_pass','recall_pass','f1_pass'])
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)   
#print(df.sort_values('f1_fail', ascending=False).head())
print(df.sort_values('f1_pass', ascending=False).head(10))
#print(df.sort_values('precision_pass', ascending=False).head(20))
#print(df.sort_values('threshold', ascending=False).head(10))

In [ ]:

plt.hist(y_scores[y_test == 1], bins=30, alpha=0.6, label="pass")
plt.hist(y_scores[y_test == 0], bins=30, alpha=0.6, label="fail")
plt.legend()
plt.xlabel("P(pass)")
plt.ylabel("Number of Boards")
plt.show()


In [ ]:
from sklearn.pipeline import Pipeline
import joblib

# Pipeline
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="liblinear",
        max_iter=1000,
        class_weight="balanced"
    ))
])

pipe.fit(X_train, y_train)

# optimal threshold for pass/fail classification
THRESHOLD_PASS = 0.43

# save the model and threshold for later use
artifact = {
    "model": pipe,
    "threshold_pass": THRESHOLD_PASS,
    "feature_cols": feature_cols
}

joblib.dump(artifact, "passfail_model.joblib")